# TransferQueue Tutorial — Basic Usage

This notebook walks through the core **Key-Value (KV) interface** of
[TransferQueue](https://github.com/Ascend/TransferQueue), an asynchronous
streaming data management module for efficient post-training workflows.

**What you will learn:**

1. Initialise TransferQueue (with Ray)
2. Store a single sample — `kv_put`
3. Store a batch of samples — `kv_batch_put`
4. Retrieve data — `kv_batch_get`
5. List stored keys & tags — `kv_list`
6. Partial-key and partial-field retrieval
7. Updating fields incrementally
8. Working with nested (variable-length) tensors
9. Storing variable-size image data
10. Storing non-tensor data (`NonTensorData` / `NonTensorStack`)
11. Multiple partitions
12. Clean up — `kv_clear` / `close`

> **Prerequisites:** `pip install TransferQueue` (or install from source).  
> Ray will be started automatically in this notebook.

## 1. Initialization

TransferQueue runs on top of [Ray](https://www.ray.io/).  
We start Ray, then call `tq.init()` with a minimal configuration that uses the
built-in **SimpleStorage** backend.

In [1]:
import ray
import torch
from omegaconf import OmegaConf
from tensordict import TensorDict
from tensordict.tensorclass import NonTensorStack

import transfer_queue as tq

ray.init(ignore_reinit_error=True)

config = OmegaConf.create(
    {
        "controller": {"polling_mode": True},
        "backend": {
            "storage_backend": "SimpleStorage",
            "SimpleStorage": {
                "total_storage_size": 200,
                "num_data_storage_units": 2,
            },
        },
    }
)

tq.init(config)
print("TransferQueue is ready!")

2026-03-27 23:06:04,557	INFO worker.py:2014 -- Started a local Ray instance. View the dashboard at http://127.0.0.1:8265 
/opt/miniconda3/envs/verl/lib/python3.11/site-packages/ray/_private/worker.py:2062: FutureWarning: Tip: In future versions of Ray, Ray will no longer override accelerator visible devices env var if num_gpus=0 or num_gpus=None (default). To enable this behavior and turn off this error message, set RAY_ACCEL_ENV_VAR_OVERRIDE_ON_ZERO=0
  warnings.warn(


(raylet) It looks like you're creating a detached actor in an anonymous namespace. In order to access this actor in the future, you will need to explicitly connect to this namespace with ray.init(namespace="798d49c0-a4e8-4877-813b-bce2d966c4eb", ...)
TransferQueue is ready!


## 2. Store a Single Sample — `kv_put`

`kv_put` stores **one** key-value pair.  
- `key` — a unique string identifier for the sample  
- `partition_id` — a logical namespace (like a table name)  
- `fields` — a `dict` of tensors **or** a `TensorDict`  
- `tag` — optional metadata dict attached to the key

In [2]:
tq.kv_put(
    key="sample_0",
    partition_id="train",
    fields={"input_ids": torch.tensor([1, 2, 3, 4])},
    tag={"source": "wikipedia", "score": 0.95},
)
print("Stored sample_0")

Stored sample_0


You can also pass a pre-built `TensorDict` directly (the batch dimension
must be 1 for `kv_put`):

In [3]:
fields_td = TensorDict(
    {
        "input_ids": torch.tensor([[5, 6, 7, 8]]),
        "attention_mask": torch.ones(1, 4, dtype=torch.long),
    },
    batch_size=1,
)

tq.kv_put(
    key="sample_1",
    partition_id="train",
    fields=fields_td,
    tag={"source": "books", "score": 0.88},
)
print("Stored sample_1")

Stored sample_1


## 3. Store a Batch of Samples — `kv_batch_put`

When you have multiple samples, `kv_batch_put` is more efficient than
calling `kv_put` in a loop.  The `fields` TensorDict must have
`batch_size == len(keys)`.

In [4]:
keys = ["batch_0", "batch_1", "batch_2"]

fields = TensorDict(
    {
        "input_ids": torch.tensor([[10, 20], [30, 40], [50, 60]]),
        "attention_mask": torch.ones(3, 2, dtype=torch.long),
    },
    batch_size=3,
)

tags = [
    {"split": "train", "idx": 0},
    {"split": "train", "idx": 1},
    {"split": "train", "idx": 2},
]

tq.kv_batch_put(keys=keys, partition_id="train", fields=fields, tags=tags)
print(f"Stored {len(keys)} samples in one call")

Stored 3 samples in one call


## 4. Retrieve Data — `kv_batch_get`

Retrieve samples by key(s).  The result is always a `TensorDict`.

In [5]:
# Retrieve a single key (pass a string)
result = tq.kv_batch_get(keys="sample_0", partition_id="train")
print("sample_0 →", result)
print("input_ids:", result["input_ids"])

sample_0 → TensorDict(
    fields={
        input_ids: Tensor(shape=torch.Size([1, 4]), device=cpu, dtype=torch.int64, is_shared=False)},
    batch_size=torch.Size([1]),
    device=None,
    is_shared=False)
input_ids: tensor([[1, 2, 3, 4]])


In [6]:
# Retrieve multiple keys at once
result = tq.kv_batch_get(keys=keys, partition_id="train")
print("batch result →", result)
print("input_ids:\n", result["input_ids"])
print("attention_mask:\n", result["attention_mask"])

batch result → TensorDict(
    fields={
        attention_mask: Tensor(shape=torch.Size([3, 2]), device=cpu, dtype=torch.int64, is_shared=False),
        input_ids: Tensor(shape=torch.Size([3, 2]), device=cpu, dtype=torch.int64, is_shared=False)},
    batch_size=torch.Size([3]),
    device=None,
    is_shared=False)
input_ids:
 tensor([[10, 20],
        [30, 40],
        [50, 60]])
attention_mask:
 tensor([[1, 1],
        [1, 1],
        [1, 1]])


## 5. List Keys & Tags — `kv_list`

`kv_list` returns a nested dict:
```
{ partition_id: { key: tag_dict, ... }, ... }
```
Pass `partition_id` to filter, or omit it to see everything.

In [7]:
info = tq.kv_list(partition_id="train")

for partition, key_tags in info.items():
    print(f"\nPartition: {partition}")
    for key, tag in key_tags.items():
        print(f"  {key}: {tag}")


Partition: train
  sample_0: {'source': 'wikipedia', 'score': 0.95}
  sample_1: {'source': 'books', 'score': 0.88}
  batch_0: {'split': 'train', 'idx': 0}
  batch_1: {'split': 'train', 'idx': 1}
  batch_2: {'split': 'train', 'idx': 2}


## 6. Partial-Key and Partial-Field Retrieval

You don't have to retrieve *all* keys or *all* fields at once.

### 6a. Partial Keys

Just pass a subset of the keys you stored.

In [8]:
partial = tq.kv_batch_get(keys=["batch_0", "batch_2"], partition_id="train")
print("Partial-key input_ids:\n", partial["input_ids"])
assert partial["input_ids"].shape[0] == 2  # only 2 rows

Partial-key input_ids:
 tensor([[10, 20],
        [50, 60]])


### 6b. Partial Fields

Use the `fields` argument to select specific columns.

In [9]:
# Retrieve only input_ids (single field)
result = tq.kv_batch_get(keys="sample_1", partition_id="train", select_fields="input_ids")
print("Fields returned:", list(result.keys()))
assert "input_ids" in result.keys()
assert "attention_mask" not in result.keys()

# Retrieve a specific set of fields
result = tq.kv_batch_get(
    keys="sample_1",
    partition_id="train",
    select_fields=["input_ids", "attention_mask"],
)
print("Fields returned:", list(result.keys()))

Fields returned: ['input_ids']
Fields returned: ['attention_mask', 'input_ids']


## 7. Updating Fields Incrementally

TransferQueue tracks each field (column) independently per key (row).  
You can **add new fields** to existing keys with another `kv_put` /
`kv_batch_put` call — the earlier fields are preserved.

In [10]:
# Add a "response" field to the batch keys
response_fields = TensorDict(
    {"response": torch.tensor([[100, 200], [300, 400], [500, 600]])},
    batch_size=3,
)

tq.kv_batch_put(keys=keys, partition_id="train", fields=response_fields)

result = tq.kv_batch_get(keys=keys, partition_id="train")
print("All fields now:", list(result.keys()))
print("response:\n", result["response"])

All fields now: ['attention_mask', 'input_ids', 'response']
response:
 tensor([[100, 200],
        [300, 400],
        [500, 600]])


## 8. Working with Nested (Variable-Length) Tensors

In many NLP and RL workloads each sample has a **different sequence length**
(e.g. generated responses).  PyTorch represents these as
[nested tensors](https://pytorch.org/docs/stable/nested.html) with the
**jagged layout** (`layout=torch.jagged`), and TransferQueue handles them
natively.

> **Note:** Because individual samples have different shapes, you must use
> `kv_batch_put` (not `kv_put`) to store nested tensors.

In [11]:
nested_keys = ["nested_0", "nested_1", "nested_2"]

# Each sample has a different sequence length
nested_responses = torch.nested.as_nested_tensor(
    [
        torch.tensor([10, 11, 12]),  # length 3
        torch.tensor([20]),  # length 1
        torch.tensor([30, 31]),  # length 2
    ],
    layout=torch.jagged,
)

nested_fields = TensorDict(
    {
        "input_ids": torch.tensor([[1, 2], [3, 4], [5, 6]]),
        "response": nested_responses,
    },
    batch_size=3,
)

tq.kv_batch_put(
    keys=nested_keys,
    partition_id="train",
    fields=nested_fields,
    tags=[{"len": 3}, {"len": 1}, {"len": 2}],
)
print("Stored 3 samples with variable-length responses")

Stored 3 samples with variable-length responses


In [12]:
# Retrieve all nested samples
result = tq.kv_batch_get(keys=nested_keys, partition_id="train")
print("input_ids:\n", result["input_ids"])
print("\nresponse (nested tensor):")
for i, sample in enumerate(result["response"]):
    print(f"  sample {i}: {sample}  (length {sample.shape[0]})")

input_ids:
 tensor([[1, 2],
        [3, 4],
        [5, 6]])

response (nested tensor):
  sample 0: tensor([10, 11, 12])  (length 3)
  sample 1: tensor([20])  (length 1)
  sample 2: tensor([30, 31])  (length 2)


Partial-key retrieval works the same way with nested tensors — only the
requested samples are returned:

In [13]:
# Retrieve only the first and last sample
partial = tq.kv_batch_get(keys=["nested_0", "nested_2"], partition_id="train")
print("Partial-key input_ids:\n", partial["input_ids"])
print("\nPartial-key responses:")
for i, sample in enumerate(partial["response"]):
    print(f"  sample {i}: {sample}")

# Verify correctness
assert torch.equal(partial["response"][0], torch.tensor([10, 11, 12]))
assert torch.equal(partial["response"][1], torch.tensor([30, 31]))
print("\nAssertions passed!")

Partial-key input_ids:
 tensor([[1, 2],
        [5, 6]])

Partial-key responses:
  sample 0: tensor([10, 11, 12])
  sample 1: tensor([30, 31])

Assertions passed!


Higher-dimensional nested tensors work too. Here each sample is a 3D
tensor with a variable first dimension (e.g. a different number of
attention heads or generated candidates):

In [14]:
nested_3d_keys = ["nd3d_0", "nd3d_1", "nd3d_2"]

nested_3d = torch.nested.as_nested_tensor(
    [
        torch.randn(2, 3, 4),  # 2 heads
        torch.randn(5, 3, 4),  # 5 heads
        torch.randn(1, 3, 4),  # 1 head
    ],
    layout=torch.jagged,
)

fields_3d = TensorDict(
    {
        "input_ids": torch.tensor([[1, 2], [3, 4], [5, 6]]),
        "hidden_states": nested_3d,
    },
    batch_size=3,
)

tq.kv_batch_put(keys=nested_3d_keys, partition_id="train", fields=fields_3d)

result_3d = tq.kv_batch_get(keys=nested_3d_keys, partition_id="train")
for i, sample in enumerate(result_3d["hidden_states"]):
    print(f"sample {i} hidden_states shape: {sample.shape}")

# Clean up nested-tensor keys
tq.kv_clear(keys=nested_keys, partition_id="train")
tq.kv_clear(keys=nested_3d_keys, partition_id="train")

sample 0 hidden_states shape: torch.Size([2, 3, 4])
sample 1 hidden_states shape: torch.Size([5, 3, 4])
sample 2 hidden_states shape: torch.Size([1, 3, 4])


## 9. Storing Variable-Size Image Data

A common multimodal scenario: each sample in a batch contains a
**different number of images**, and each image has a **different
resolution**.  We can model this with a list of nested tensors — one
nested tensor per sample — wrapped inside a `TensorDict`.

Since the data is doubly ragged (variable count *and* variable size),
we store each image as a flattened 1-D tensor and pack all images per
sample into a single jagged nested tensor.  This way every sample is
one element of the batch, yet images retain their individual sizes.

In [15]:
image_keys = ["img_sample_0", "img_sample_1", "img_sample_2"]

# Sample 0: 2 images — 3×32×32 (RGB, 32×32) and 3×64×64
sample_0_images = [torch.randn(3, 32, 32), torch.randn(3, 64, 64)]

# Sample 1: 1 image — 3×48×48
sample_1_images = [torch.randn(3, 48, 48)]

# Sample 2: 3 images — 3×16×16, 3×24×24, 3×32×64
sample_2_images = [torch.randn(3, 16, 16), torch.randn(3, 24, 24), torch.randn(3, 32, 64)]


# Flatten each image to 1-D so they can live in a single jagged nested tensor per sample
def flatten_images(images):
    return torch.cat([img.flatten() for img in images])


pixel_data = torch.nested.as_nested_tensor(
    [
        flatten_images(sample_0_images),  # 3*32*32 + 3*64*64 = 15360
        flatten_images(sample_1_images),  # 3*48*48            = 6912
        flatten_images(sample_2_images),
    ],  # 3*16*16 + 3*24*24 + 3*32*64 = 8736
    layout=torch.jagged,
)

# Store the number of pixels per image so we can reconstruct later
image_shapes = torch.nested.as_nested_tensor(
    [
        torch.tensor([[3, 32, 32], [3, 64, 64]]),  # 2 images
        torch.tensor([[3, 48, 48]]),  # 1 image
        torch.tensor([[3, 16, 16], [3, 24, 24], [3, 32, 64]]),  # 3 images
    ],
    layout=torch.jagged,
)

fields_img = TensorDict(
    {
        "prompt": torch.tensor([[101, 102], [201, 202], [301, 302]]),
        "pixel_data": pixel_data,
        "image_shapes": image_shapes,
    },
    batch_size=3,
)

tags_img = [
    {"num_images": 2},
    {"num_images": 1},
    {"num_images": 3},
]

tq.kv_batch_put(keys=image_keys, partition_id="train", fields=fields_img, tags=tags_img)
print("Stored 3 samples with variable numbers of variable-size images")

Stored 3 samples with variable numbers of variable-size images


In [16]:
# Retrieve and reconstruct the images
result_img = tq.kv_batch_get(keys=image_keys, partition_id="train")

for i in range(result_img.batch_size[0]):
    shapes = result_img["image_shapes"][i]  # (num_images, 3) tensor
    pixels = result_img["pixel_data"][i]  # flat 1-D tensor of all pixels
    num_images = shapes.shape[0]

    offset = 0
    reconstructed = []
    for j in range(num_images):
        c, h, w = shapes[j].tolist()
        numel = c * h * w
        img = pixels[offset : offset + numel].reshape(c, h, w)
        reconstructed.append(img)
        offset += numel

    print(f"Sample {i}: {num_images} image(s) → {[tuple(img.shape) for img in reconstructed]}")

# Clean up
tq.kv_clear(keys=image_keys, partition_id="train")

Sample 0: 2 image(s) → [(3, 32, 32), (3, 64, 64)]
Sample 1: 1 image(s) → [(3, 48, 48)]
Sample 2: 3 image(s) → [(3, 16, 16), (3, 24, 24), (3, 32, 64)]


## 10. Storing Non-Tensor Data — `NonTensorData` / `NonTensorStack`

Not every field is a numeric tensor.  Prompts, file paths, JSON metadata,
or arbitrary Python objects can be stored as **non-tensor data** using
tensordict's `NonTensorData` and `NonTensorStack`.

- `NonTensorData` wraps a **single** Python object (string, dict, list, …).
- `NonTensorStack` wraps a **batch** of Python objects — one per sample.

TransferQueue serialises them transparently alongside regular tensors.

In [17]:
nt_keys = ["nt_sample_0", "nt_sample_1", "nt_sample_2"]

# Build a TensorDict that mixes tensors with non-tensor data
nt_fields = TensorDict(
    {
        "input_ids": torch.tensor([[1, 2, 3], [4, 5, 6], [7, 8, 9]]),
        # NonTensorStack: one string per sample in the batch
        "prompt_text": NonTensorStack(
            "Summarise the following article.",
            "Translate to French:",
            "Write a poem about rain.",
        ),
        # You can also store richer Python objects (dicts, lists, …)
        "metadata": NonTensorStack(
            {"source": "wiki", "lang": "en"},
            {"source": "books", "lang": "fr"},
            {"source": "user", "lang": "en"},
        ),
    },
    batch_size=3,
)

tq.kv_batch_put(keys=nt_keys, partition_id="train", fields=nt_fields)
print("Stored 3 samples with non-tensor fields")

Stored 3 samples with non-tensor fields


In [18]:
# Retrieve and inspect non-tensor fields
result_nt = tq.kv_batch_get(keys=nt_keys, partition_id="train")

print("input_ids:\n", result_nt["input_ids"])

print("\nprompt_text (NonTensorStack):")
for i, text in enumerate(list(result_nt["prompt_text"])):
    print(f"  [{i}] {text!r}")

print("\nmetadata (NonTensorStack of dicts):")
for i, meta in enumerate(list(result_nt["metadata"])):
    print(f"  [{i}] {meta}")

# You can also add a NonTensorData field to a single key via kv_put
tq.kv_put(
    key="single_nt",
    partition_id="train",
    fields={"label": NonTensorStack("positive"), "score": torch.tensor([0.99])},
)
single = tq.kv_batch_get(keys="single_nt", partition_id="train")
print(f"\nSingle-key non-tensor field: label={list(single['label'])}")

# Clean up
tq.kv_clear(keys=nt_keys, partition_id="train")
tq.kv_clear(keys="single_nt", partition_id="train")

input_ids:
 tensor([[1, 2, 3],
        [4, 5, 6],
        [7, 8, 9]])

prompt_text (NonTensorStack):
  [0] 'Summarise the following article.'
  [1] 'Translate to French:'
  [2] 'Write a poem about rain.'

metadata (NonTensorStack of dicts):
  [0] {'source': 'wiki', 'lang': 'en'}
  [1] {'source': 'books', 'lang': 'fr'}
  [2] {'source': 'user', 'lang': 'en'}

Single-key non-tensor field: label=[['positive']]


## 11. Multiple Partitions

Partitions provide logical isolation — the same key name can exist in
different partitions without conflict.

In [19]:
tq.kv_put(
    key="val_sample_0",
    partition_id="validation",
    fields={"input_ids": torch.tensor([99, 98, 97])},
    tag={"split": "val"},
)

all_info = tq.kv_list()  # no partition_id → list everything
print("All partitions:", list(all_info.keys()))

All partitions: ['train', 'validation']


## 12. Clean Up — `kv_clear` and `close`

Remove specific keys with `kv_clear`, then shut down the system with `tq.close()`.

In [20]:
# Clear individual keys
tq.kv_clear(keys="sample_0", partition_id="train")
tq.kv_clear(keys="sample_1", partition_id="train")
tq.kv_clear(keys=keys, partition_id="train")
tq.kv_clear(keys="val_sample_0", partition_id="validation")
print("All keys cleared.")

remaining = tq.kv_list()
total_keys = sum(len(v) for v in remaining.values())
print(f"Remaining keys across all partitions: {total_keys}")

All keys cleared.
Remaining keys across all partitions: 0


In [21]:
tq.close()
ray.shutdown()
print("TransferQueue and Ray shut down.")

TransferQueue and Ray shut down.


---

## Summary

| Operation | Function | Notes |
|---|---|---|
| Init | `tq.init(config)` | Call once; subsequent processes auto-connect |
| Put single | `tq.kv_put(key, partition_id, fields, tag)` | `fields` can be a plain dict |
| Put batch | `tq.kv_batch_put(keys, partition_id, fields, tags)` | `fields` must be a `TensorDict` |
| Get | `tq.kv_batch_get(keys, partition_id, select_fields=None)` | Returns a `TensorDict` |
| List | `tq.kv_list(partition_id=None)` | Returns `{partition: {key: tag}}` |
| Clear | `tq.kv_clear(keys, partition_id)` | Removes keys + data |
| Close | `tq.close()` | Tears down controller & storage |

For **async** variants, use `async_kv_put`, `async_kv_batch_put`,
`async_kv_batch_get`, `async_kv_list`, and `async_kv_clear`.

For low-level, metadata-based access, see `tq.get_client()` and the
[official tutorials](https://github.com/Ascend/TransferQueue/tree/main/tutorial).